<a href="https://colab.research.google.com/github/LucianoBV/AprendizajeAutomatico2/blob/main/Luciano_Vargas_Parcial2_Spark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de ventas con PySpark y Plotly
**Alumno:** Luciano Vargas

**Materia:** Procesamiento de Datos

**Parcial 2**

**Modalidad:** individual

**Objetivo:** procesar un conjunto de datos de ventas con Spark para analizar su evolución mensual, productos, países y días de la semana.

**Enfoque temático:** análisis exploratorio estructurado con DataFrames de Spark y operaciones nativas que pueden ejecutarse en paralelo. **Bloque opcional:** visualización interactiva con Plotly.

**Pipeline ETL:** descargar → cargar con Spark → explorar → limpiar → transformar → guardar → analizar y visualizar.



## 1. Preparar el entorno
Se instalan PySpark y Plotly. Pandas se utiliza únicamente al final, mediante toPandas(), para pasar tablas pequeñas a los gráficos. No se usa para cargar ni limpiar el dataset.



In [1]:
%pip -q install pyspark plotly pandas

In [2]:
from importlib.metadata import version

for paquete in ["pyspark", "plotly", "pandas"]:
    print(f"{paquete}=={version(paquete)}")

pyspark==4.0.5
plotly==5.24.1
pandas==2.2.3


In [3]:
# IMPORTAMOS LIBRERIAS
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import plotly.express as px

In [4]:
# CONFIGURAMOS LA SESION
spark = SparkSession.builder.master("local[2]").appName("AnalisisVentas").getOrCreate()

SparkSession inicia el acceso a Spark.

Local[2] permite trabajar con dos hilos en una sola computadora. Es ejecución local con paralelismo; no significa que haya dos computadoras conectadas.

## 2. Extraer: descargar y leer el dataset
Se utiliza **Online Retail**, de Daqing Chen (2015), publicado en UCI: https://doi.org/10.24432/C5BW33, licencia CC BY 4.0.

Son líneas de facturas de una tienda del Reino Unido, entre diciembre de 2010 y diciembre de 2011. Una factura puede tener varias filas. Los precios están en libras esterlinas (GBP).

El original de UCI está en Excel. Aquí se descarga una copia CSV pública del repositorio docente dbdmg/data-science-lab, fijada a una versión para que la URL no cambie con nuevas ediciones.

La descarga la realiza wget, un comando del entorno Colab. Spark lee el archivo descargado con spark.read.csv. Son dos pasos distintos. -O indica el nombre del archivo local y -q evita mostrar el progreso de la descarga.

In [5]:
url = "https://raw.githubusercontent.com/dbdmg/data-science-lab/fae68c993786998d03f5bfa6959ee29e09198715/datasets/online_retail.csv"
!wget -q -O ventas.csv "{url}"

In [6]:
df = spark.read.csv("ventas.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()
print("Filas originales:", df.count())

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
only showing top 5 rows
root
 |-- InvoiceNo: string (nullable = true)
 |-

header=True usa la primera fila como nombres de columnas. inferSchema=True permite que Spark identifique sus tipos; después se verifican y convierten los campos necesarios. show(5) muestra cinco filas, printSchema() muestra los tipos y count() cuenta los registros.

| Columna | Significado |
|---|---|
| InvoiceNo | Número de factura; si empieza por C, indica anulación. |
| StockCode | Código del producto. |
| Description | Descripción del producto. |
| Quantity | Unidades de esa línea de factura. |
| InvoiceDate | Fecha y hora. |
| UnitPrice | Precio por unidad en GBP. |
| CustomerID | Identificador del cliente, cuando está disponible. |
| Country | País del cliente. |

## 3. Explorar los datos
Se cuentan los nulos de cada columna y se observan las variables numéricas antes de limpiar. La repetición con for realiza la misma consulta para cada columna.

Los nulos de CustomerID no se eliminan: este proyecto estudia ventas por producto, fecha y país, no perfiles individuales de clientes.

In [ ]:
for columna in df.columns:
    cantidad = df.filter(F.col(columna).isNull()).count()
    print(columna, ":", cantidad, "nulos")

df.select("Quantity", "UnitPrice").describe().show()

InvoiceNo : 0 nulos
StockCode : 0 nulos
Description : 1454 nulos
Quantity : 0 nulos
InvoiceDate : 0 nulos


## 4. Transformar: convertir tipos y limpiar
La fecha del CSV usa mes/día/año y hora. to_timestamp la convierte a un tipo temporal; una conversión fallida produce un nulo. Cantidad y precio se convierten a tipos numéricos.

Después se eliminan nulos únicamente en las columnas necesarias. Cada paso imprime la cantidad de filas que conserva.

In [ ]:
limpios = df.withColumn("InvoiceDate", F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
limpios = limpios.withColumn("Quantity", F.col("Quantity").cast("int"))
limpios = limpios.withColumn("UnitPrice", F.col("UnitPrice").cast("decimal(12,3)"))
limpios = limpios.dropna(subset=["InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice", "Country"])
print("Filas con campos necesarios completos:", limpios.count())

withColumn crea o reemplaza una columna. cast cambia su tipo. decimal(12,3) permite 12 cifras totales, de las cuales 3 son decimales: se utiliza para conservar los precios monetarios de la fuente.

In [ ]:
limpios = limpios.filter(~F.col("InvoiceNo").startswith("C"))
limpios = limpios.filter((F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
print("Filas de compras positivas:", limpios.count())

limpios = limpios.dropDuplicates()
print("Filas después de eliminar duplicados:", limpios.count())

filter conserva las filas que cumplen una condición. ~ niega una condición y & combina dos condiciones que deben cumplirse al mismo tiempo.

Las anulaciones, cantidades negativas y precios no positivos quedan fuera porque el objetivo es estudiar **compras positivas**, no ventas netas. Un precio cero podría ser un regalo; no se lo declara automáticamente como un error.

dropDuplicates() elimina filas completamente iguales. Es una decisión del proyecto: sin un identificador único de línea, algunas repeticiones podrían ser legítimas. Los conteos anteriores permiten ver el impacto. El archivo original descargado permanece disponible.

## 5. Crear las variables para el análisis
Se calcula el importe de cada línea y se obtiene el mes y el día de la semana. Los importes se conservan con los decimales de la fuente y se redondean a dos decimales al presentar los totales.

In [ ]:
ventas = limpios.withColumn("Importe", F.col("Quantity") * F.col("UnitPrice"))
ventas = ventas.withColumn("Mes", F.date_format("InvoiceDate", "yyyy-MM"))
ventas = ventas.withColumn("DiaSemana", F.dayofweek("InvoiceDate"))
ventas.select("InvoiceNo", "Quantity", "UnitPrice", "Importe", "Mes", "DiaSemana").show(5)

## 6. Cargar: guardar los datos procesados
Spark guarda en Parquet, un formato por columnas que conserva tipos de datos. Se obtiene una carpeta con archivos, no un único archivo. overwrite reemplaza esa salida al volver a ejecutar.

Se vuelve a leer la salida para que las consultas siguientes trabajen sobre el resultado del ETL.

In [ ]:
ventas.write.mode("overwrite").parquet("resultados/ventas_limpias")
ventas = spark.read.parquet("resultados/ventas_limpias")
print("Filas guardadas y leídas:", ventas.count())

## 7. Análisis exploratorio con Spark
Se responden cinco preguntas. Todas las agrupaciones y cálculos se realizan en Spark, antes de visualizar.

groupBy forma grupos, agg calcula indicadores, sum suma, avg promedia, alias pone un nombre al resultado y orderBy ordena. countDistinct cuenta valores diferentes, por ejemplo facturas, en lugar de contar renglones.

### 7.1. ¿Cómo evolucionan las compras por mes?
Se calculan importe y número de facturas por mes.

In [ ]:
por_mes = ventas.groupBy("Mes").agg(
    F.round(F.sum("Importe"), 2).alias("Ventas_GBP"),
    F.countDistinct("InvoiceNo").alias("Facturas")
).orderBy("Mes")
por_mes.show()

### 7.2. ¿Cuáles son los diez productos con más unidades vendidas?
Se agrupa por código, porque las descripciones pueden variar para un mismo producto. Se muestra también el importe para distinguir cantidad vendida de valor monetario.

In [ ]:
por_producto = ventas.groupBy("StockCode").agg(
    F.sum("Quantity").alias("Unidades"),
    F.round(F.sum("Importe"), 2).alias("Ventas_GBP")
).orderBy(F.desc("Unidades"), "StockCode").limit(10)
por_producto.show()

### 7.3. ¿Qué países concentran el importe de las compras?
Se muestran los diez países con mayor importe. Es un ranking parcial; no debe interpretarse como la distribución completa de todos los países.

In [ ]:
por_pais = ventas.groupBy("Country").agg(
    F.round(F.sum("Importe"), 2).alias("Ventas_GBP")
).orderBy(F.desc("Ventas_GBP"), "Country").limit(10)
por_pais.show()

### 7.4. ¿Cuánto se compra en promedio por factura según el día de la semana?
Primero se suman los renglones de cada factura. Después se calcula el importe promedio de las facturas por día de la semana. Así se evita confundir importe medio de una línea con importe medio de una factura.

Spark numera los días así: 1 = domingo, 2 = lunes, 3 = martes, 4 = miércoles, 5 = jueves, 6 = viernes, 7 = sábado. Un día sin registros no prueba por sí solo que el comercio estuviera cerrado.

In [ ]:
facturas = ventas.groupBy("InvoiceNo", "DiaSemana").agg(F.sum("Importe").alias("TotalFactura"))
por_dia = facturas.groupBy("DiaSemana").agg(
    F.round(F.avg("TotalFactura"), 2).alias("CompraMedia_GBP"),
    F.count("InvoiceNo").alias("Facturas")
).orderBy("DiaSemana")
por_dia.show()

### 7.5. ¿Cómo se relacionan mes, país e importe?
Se cruzan dos variables de agrupación (mes y país), para observar diferencias temporales entre mercados. Esta tabla complementa los rankings globales.

In [ ]:
por_mes_pais = ventas.groupBy("Mes", "Country").agg(
    F.round(F.sum("Importe"), 2).alias("Ventas_GBP")
).orderBy("Mes", "Country")
por_mes_pais.show(10)

### Paralelización nativa
Las consultas anteriores usan DataFrames y funciones de Spark, como en el enfoque de Spark nativo. Las agrupaciones pueden redistribuir datos entre particiones. El número de particiones no representa el número de computadoras.

explain() muestra el plan de una consulta. Exchange indica una redistribución; las agregaciones calculan los resultados por grupo. filter y withColumn son transformaciones; acciones como show, count y la escritura solicitan la ejecución.

In [ ]:
print("Particiones del DataFrame:", ventas.rdd.getNumPartitions())
por_mes.explain()

## 8. Visualización interactiva con Plotly
No se limita a un único gráfico: se presentan cinco vistas complementarias. Cada gráfico permite consultar valores al pasar el cursor y utilizar sus controles. No se implementan filtros compartidos entre gráficos.

Solo se convierten a Pandas las tablas pequeñas resultantes de Spark, no las más de 500.000 líneas originales. astype(float) adapta los importes decimales a números compatibles con Plotly.

In [ ]:
tabla_mes = por_mes.toPandas()
tabla_mes["Ventas_GBP"] = tabla_mes["Ventas_GBP"].astype(float)
fig_mes = px.line(tabla_mes, x="Mes", y="Ventas_GBP", markers=True,
                  title="Compras por mes en GBP — diciembre de 2011 incompleto")
fig_mes.show()

In [ ]:
tabla_producto = por_producto.toPandas()
fig_producto = px.bar(tabla_producto, x="StockCode", y="Unidades",
                       title="Diez productos con más unidades vendidas")
fig_producto.update_xaxes(type="category")
fig_producto.show()

In [ ]:
tabla_pais = por_pais.toPandas()
tabla_pais["Ventas_GBP"] = tabla_pais["Ventas_GBP"].astype(float)
fig_pais = px.bar(tabla_pais, x="Country", y="Ventas_GBP",
                   title="Diez países con mayor importe de compras (GBP)")
fig_pais.show()

In [ ]:
tabla_dia = por_dia.toPandas()
tabla_dia["CompraMedia_GBP"] = tabla_dia["CompraMedia_GBP"].astype(float)
fig_dia = px.bar(tabla_dia, x="DiaSemana", y="CompraMedia_GBP",
                  title="Compra media por factura (GBP) — 1: domingo, 2: lunes, …, 7: sábado")
fig_dia.show()

In [ ]:
tabla_mes_pais = por_mes_pais.toPandas()
tabla_mes_pais["Ventas_GBP"] = tabla_mes_pais["Ventas_GBP"].astype(float)
fig_mes_pais = px.line(tabla_mes_pais, x="Mes", y="Ventas_GBP", color="Country",
                       title="Evolución de compras por país (GBP)")
fig_mes_pais.show()

## 9. Guardar las consultas y el panel visual
Las consultas agregadas también se guardan en Parquet. Los cinco gráficos se reúnen en un HTML que se abre con un navegador y funciona sin Internet. El primer gráfico incluye el código de Plotly; los demás reutilizan ese mismo código.

with open abre el archivo para escribir, y for recorre la lista de gráficos. to_html convierte cada gráfico a un fragmento HTML. No se utilizan otras librerías para construir el panel.

In [ ]:
por_mes.write.mode("overwrite").parquet("resultados/ventas_por_mes")
por_producto.write.mode("overwrite").parquet("resultados/top_productos")
por_pais.write.mode("overwrite").parquet("resultados/top_paises")
por_dia.write.mode("overwrite").parquet("resultados/compra_media_por_dia")
por_mes_pais.write.mode("overwrite").parquet("resultados/ventas_por_mes_pais")

In [ ]:
graficos = [fig_mes, fig_producto, fig_pais, fig_dia, fig_mes_pais]
with open("resultados/panel_ventas.html", "w", encoding="utf-8") as archivo:
    archivo.write("<!doctype html><html lang='es'><meta charset='utf-8'><title>Análisis de ventas</title><body><h1>Análisis de ventas — Luciano Vargas</h1><p>Compras positivas; importes en GBP.</p>")
    for numero, grafico in enumerate(graficos):
        archivo.write(grafico.to_html(full_html=False, include_plotlyjs=(numero == 0)))
    archivo.write("</body></html>")
print("Panel guardado en resultados/panel_ventas.html")

## 10. Alcance y conclusiones
El flujo usa Spark para extraer del CSV, limpiar, transformar, guardar y analizar. El archivo se descarga previamente con un comando del entorno. Plotly representa los resultados agregados y Pandas sirve de puente para esas tablas pequeñas.

El proyecto es descriptivo: no utiliza machine learning ni pronostica ventas. Su enfoque temático es el análisis exploratorio con procesamiento nativo de Spark; las cinco consultas y sus visualizaciones forman parte de un mismo flujo.

Los importes representan compras positivas después de la limpieza y deduplicación. No son ventas netas de devoluciones. Se conservan ventas sin identificación de cliente, porque no se estudian perfiles personales. El dataset es histórico, de una sola empresa, y diciembre de 2011 es incompleto. El modo local no demuestra escalabilidad real entre máquinas.

Los hallazgos concretos de esta ejecución se presentan a continuación.

### Resultados de la ejecución
Se leyeron **541.909** líneas y se conservaron **524.878** después de la limpieza.

- El mes con mayor importe en el período analizado fue **2011-11**.
- El país con mayor importe fue **United Kingdom**.
- El producto con más unidades fue **23843**.

Los resultados corresponden al universo definido en este notebook. No permiten afirmar relaciones causales ni anticipar ventas futuras.